# 02 — Baseline Error Analysis

Run all baselines on the test set and analyze their errors.
- LAI, Dictionary, Rule-based, Retrieval
- Both directions: dialect→standard and standard→dialect
- 20 failure cases per baseline with linguistic explanations

In [ ]:
import json
import sys
from pathlib import Path
from collections import Counter

import pandas as pd

sys.path.insert(0, str(Path("..").resolve()))
from src.model.config import DataConfig

cfg = DataConfig()

## Run Baselines

In [ ]:
from src.baselines import lai_baseline, dict_baseline, rule_baseline

test_path = cfg.processed_dir / "test.jsonl"
train_path = cfg.processed_dir / "train.jsonl"
out_dir = Path("../results/metrics")

# LAI
lai_results = lai_baseline.run_on_file(test_path, out_dir / "lai_predictions.jsonl")
print(f"LAI: {len(lai_results)} predictions")

# Rule-based
rule_results = rule_baseline.run_on_file(test_path, out_dir / "rule_predictions.jsonl")
print(f"Rule-based: {len(rule_results)} predictions")

## Evaluate All Baselines

In [ ]:
from src.evaluation.metrics import compute_bleu, compute_rouge_l, compute_dfr

def eval_baseline(results, name):
    by_task = {}
    for r in results:
        task = r["task"]
        by_task.setdefault(task, {"preds": [], "refs": [], "regions": []})
        by_task[task]["preds"].append(r["prediction"])
        by_task[task]["refs"].append(r["target"])
        by_task[task]["regions"].append(r.get("region", ""))

    print(f"\n{'='*50}")
    print(f"Baseline: {name}")
    print(f"{'='*50}")
    for task, data in by_task.items():
        bleu = compute_bleu(data["preds"], data["refs"])
        rouge = compute_rouge_l(data["preds"], data["refs"])
        print(f"  {task}: BLEU={bleu:.2f}, ROUGE-L={rouge:.4f}")
        if task.startswith("std2dialect"):
            dfr = compute_dfr(data["preds"], data["refs"], data["regions"])
            print(f"    DFR: {dfr}")

eval_baseline(lai_results, "LAI")
eval_baseline(rule_results, "Rule-based")

## Error Analysis: 20 Failures per Baseline

For each baseline, show cases where prediction ≠ reference,
with a brief linguistic explanation.

In [ ]:
def show_failures(results, name, max_show=20):
    failures = [r for r in results if r["prediction"].strip() != r["target"].strip()]
    print(f"\n{'='*60}")
    print(f"{name} — {len(failures)} failures out of {len(results)} total")
    print(f"{'='*60}")
    for i, r in enumerate(failures[:max_show]):
        print(f"\n[{i+1}] Task: {r['task']}, Region: {r.get('region', 'N/A')}")
        print(f"  SRC: {r['source']}")
        print(f"  REF: {r['target']}")
        print(f"  PRD: {r['prediction']}")
        # Highlight difference
        src_words = set(r['source'].lower().split())
        ref_words = set(r['target'].lower().split())
        pred_words = set(r['prediction'].lower().split())
        missed = ref_words - pred_words
        extra = pred_words - ref_words
        if missed:
            print(f"  MISSED: {missed}")
        if extra:
            print(f"  EXTRA:  {extra}")

show_failures(lai_results, "LAI Baseline")
show_failures(rule_results, "Rule-based Baseline")

## Error Type Summary

In [ ]:
def categorize_error(r):
    """Simple heuristic categorization."""
    pred = r["prediction"].lower()
    ref = r["target"].lower()
    src = r["source"].lower()
    if pred == src:
        return "no_change"
    elif pred == ref:
        return "correct"
    else:
        # Check if it's a partial match
        pred_words = set(pred.split())
        ref_words = set(ref.split())
        overlap = len(pred_words & ref_words) / max(len(ref_words), 1)
        if overlap > 0.8:
            return "partial_match"
        elif overlap > 0.5:
            return "moderate_diff"
        else:
            return "major_diff"

for name, results in [("LAI", lai_results), ("Rule-based", rule_results)]:
    cats = Counter(categorize_error(r) for r in results)
    print(f"\n{name}: {dict(cats)}")